In [ ]:
# pip install opencv-python-headless
# pip install pillow
# pip install pytesseract

In [2]:
import json
import pandas as pd

data = json.load(
    open(
        "/home/rsoko/private/001-wedding-venue/data/processed/adobe_extracted/ Monserate Winery (11)-merged/structuredData.json"
    )
)
page = ""
for ele in data["elements"]:
    if "Page" in ele.keys():
        page = ele["Page"]

    elif ("Text" in ele.keys()) and ("Figure" not in ele["Path"]):
        df = pd.DataFrame({"text": ele["Text"]}, index=[0])


{'version': {'json_export': '210',
  'page_segmentation': '55',
  'schema': '1.1.0',
  'structure': '1.1125.0',
  'table_structure': '5'},
 'extended_metadata': {'ID_instance': '4D 04 66 82 51 BB B2 11 0A 00 1E 69 5C 61 78 4F ',
  'ID_permanent': '33 A5 0E 6D CF 57 63 F8 8B 97 3B 18 A0 63 7F 73 ',
  'has_acroform': False,
  'has_embedded_files': False,
  'is_XFA': False,
  'is_certified': False,
  'is_encrypted': False,
  'is_digitally_signed': False,
  'language': 'en',
  'page_count': 31,
  'pdf_version': '1.6',
  'pdfa_compliance_level': '',
  'pdfua_compliance_level': ''},
 'elements': [{'Bounds': [-0.7674102783203125,
    474.1995391845703,
    612.7766265869141,
    1087.7435913085938],
   'ObjectID': 2015,
   'Page': 0,
   'Path': '//Document/Figure',
   'attributes': {'BBox': [0.0,
     474.97299999999814,
     611.9749999999767,
     799.9199999999837],
    'Placement': 'Block'},
   'filePaths': ['figures/fileoutpart0.png']},
  {'Bounds': [-0.7674102783203125,
    474.19953918

In [1]:
from tqdm import tqdm
import cv2
import numpy as np
from PIL import Image
import pytesseract
from pathlib import Path

import pandas as pd


def photo_properties(image_path) -> dict:
    # Read image using PIL first (for OCR)
    pil_image = Image.open(image_path)

    # Get OCR data including bounding boxes
    ocr_data = pytesseract.image_to_data(
        pil_image, output_type=pytesseract.Output.DATAFRAME
    )

    # Calculate total area of text bounding boxes
    total_text_area = 0
    try:
        if not ocr_data.empty:
            # Filter out empty text entries and calculate areas
            valid_boxes = ocr_data[
                ocr_data["text"].notna() & (ocr_data["text"].str.strip() != "")
            ]
            total_text_area = sum(
                row["width"] * row["height"] for _, row in valid_boxes.iterrows()
            )
    except Exception as e:
        pass
    # Get image dimensions for total area
    width, height = pil_image.size
    total_image_area = width * height

    # Calculate text density as ratio of text area to image area
    text_density = total_text_area / total_image_area if total_image_area > 0 else 0

    # Convert to OpenCV format for edge detection
    cv_image = cv2.imread(str(image_path))
    gray = cv2.cvtColor(cv_image, cv2.COLOR_BGR2GRAY)

    # Image properties
    height, width = gray.shape
    total_pixels = height * width

    # Edge detection
    edges = cv2.Canny(gray, 100, 200)
    edge_pixels = np.count_nonzero(edges)
    edge_density = edge_pixels / total_pixels

    # Analyze color variation
    color_std = np.std(cv_image, axis=(0, 1)).mean()

    return {
        "width": width,
        "height": height,
        "total_pixels": total_pixels,
        "text_density": text_density,
        "color_std": color_std,
        "edge_density": edge_density,
    }


def predict_image_quality(properties: dict) -> bool:
    X = pd.DataFrame(
        {
            "width": properties["width"],
            "height": properties["height"],
            "total_pixels": properties["total_pixels"],
            "text_density": properties["text_density"],
            "color_std": properties["color_std"],
            "edge_density": properties["edge_density"],
        },
        index=[0],
    )
    return bool(model.predict(X)[0])


def is_photo(image_path, text_threshold=0.05, color_std_threshold=30):
    """
    Determines if an image is likely a photo rather than text/logo/menu.

    Args:
        image_path: Path to the image file
        text_threshold: Maximum ratio of text area to consider as photo
        edge_threshold: Threshold for edge density to distinguish photos from graphics

    Returns:
        bool: True if the image is likely a photo, False otherwise
    """
    properties = photo_properties(image_path)
    # Classification rules
    is_likely_photo = (
        properties["text_density"] < text_threshold  # Not too much text
        and properties["color_std"]
        > color_std_threshold  # Has reasonable color variation
    )

    return is_likely_photo


def filter_photos(input_directory):
    """
    Processes a directory of images and copies only the photos to the output directory.

    Args:
        input_directory: Path to directory containing input images
        output_directory: Path to directory where photos should be copied
    """
    input_path = Path(input_directory)
    output_path = input_path.with_name(input_path.stem + "_accepted")
    output_path.mkdir(parents=True, exist_ok=True)
    rejected_path = input_path.with_name(input_path.stem + "_rejected")
    rejected_path.mkdir(parents=True, exist_ok=True)

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    if not Path(input_directory).exists():
        return
    for image_file in tqdm(list(input_path.iterdir())):
        if image_file.suffix.lower() in image_extensions:
            try:
                if is_photo(image_file):
                    Image.open(image_file).save(output_path / image_file.name)
                else:
                    Image.open(image_file).save(rejected_path / image_file.name)
            except Exception as e:
                pass


# Example usage
if __name__ == "__main__":
    filter_photos(
        "../data/processed/adobe_extracted/ Monserate Winery (11)-merged/figures",
    )

100%|██████████| 2/2 [00:00<00:00, 9608.94it/s]


In [2]:
import pandas as pd
from pathlib import Path

img_dir = Path("../data/processed/train")
bad_img_dir = Path("../data/processed/rejected_pdf_images")
df = pd.DataFrame()
for img in tqdm(list(img_dir.iterdir())):
    if img.name in list(bad_img_dir.iterdir()):
        good_img = False
    else:
        good_img = True
    try:
        properties = photo_properties(img)
    except Exception as e:
        continue
    properties.update({"img": img.name, "good_img": good_img})
    df = pd.concat([df, pd.DataFrame(properties, index=[len(df)])], ignore_index=True)
img = pd.Series(list(x.name for x in img_dir.iterdir()))
good_img = ~img.isin(list(x.name for x in bad_img_dir.iterdir()))
df["good_img"] = good_img
df["img"] = img
df.good_img.value_counts()

df

100%|██████████| 473/473 [13:01<00:00,  1.65s/it]


,width,height,total_pixels,text_density,color_std,edge_density,img,good_img
0,13,13,169,0.000000,0.000000,0.000000,2024 Queen Mary Ceremony Locations-merged_file...,False
1,1525,441,672525,0.000000,67.095970,0.117889,2023 Continental Catering Full Service Menu_fi...,True
2,2550,1176,2998800,0.000000,53.423644,0.037985,2025 Alta Vista Wedding Packages_fileoutpart14...,True
3,1032,1117,1152744,0.000000,56.640216,0.047789,Monserate Winery (11)-merged_fileoutpart58.png,True
4,725,1449,1050525,0.000000,66.490859,0.065672,Monserate Winery (11)-merged_fileoutpart28.png,True
...,...,...,...,...,...,...,...,...
468,1051,1670,1755170,0.000000,60.955133,0.090637,2024 Queen Mary Ceremony Locations-merged_file...,True
469,2550,2746,7002300,0.029906,32.266259,0.005549,_2024 Weddings at Lake Arrowhead~-merged_fileo...,False
470,2568,3299,8471832,0.000000,79.503724,0.015911,2023 Estancia La Jolla Wedding Brochure-merged...,True
471,2402,1893,4546986,0.000000,58.661994,0.094875,2024 L_Auberge del mar_fileoutpart2.png,True


In [5]:
import itables

itables.show(df)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

# Prepare features and target
features = [
    "width",
    "height",
    "total_pixels",
    "text_density",
    "color_std",
    "edge_density",
]
X = df[features]
y = df["good_img"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a random forest model
model = RandomForestClassifier(
    n_estimators=100,  # number of trees
    max_depth=None,  # maximum depth of trees
    min_samples_split=2,
    random_state=42,
)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Print feature importance
feature_importance = pd.DataFrame(
    {"feature": features, "importance": model.feature_importances_}
)
print("\nFeature Importance:")
print(feature_importance.sort_values("importance", ascending=False))


# Function to predict if a new image is good
def predict_image_quality(properties: dict) -> bool:
    X = pd.DataFrame(
        {
            "width": properties["width"],
            "height": properties["height"],
            "total_pixels": properties["total_pixels"],
            "text_density": properties["text_density"],
            "color_std": properties["color_std"],
            "edge_density": properties["edge_density"],
        },
        index=[0],
    )
    return bool(model.predict(X)[0])

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.93      0.96        41
        True       0.95      1.00      0.97        54

    accuracy                           0.97        95
   macro avg       0.97      0.96      0.97        95
weighted avg       0.97      0.97      0.97        95


Feature Importance:
        feature  importance
1        height    0.295951
2  total_pixels    0.194837
4     color_std    0.158502
0         width    0.145348
3  text_density    0.112306
5  edge_density    0.093056


In [8]:
import pickle

with open("model.bin", "wb") as f:
    pickle.dump(model, f)

In [ ]:
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

MODEL_PATH=Path("model.bin")

def load_is_photo_classifier()->RandomForestClassifier:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    return model

def is_photo(image_path: Path | str):
    

In [9]:
with open("model.bin", "rb") as f:  # 'rb' for binary read mode
    model = pickle.load(f)

In [10]:
y_pred = model.predict(X_test)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Print feature importance
feature_importance = pd.DataFrame(
    {"feature": features, "importance": model.feature_importances_}
)
print("\nFeature Importance:")
print(feature_importance.sort_values("importance", ascending=False))

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.93      0.96        41
        True       0.95      1.00      0.97        54

    accuracy                           0.97        95
   macro avg       0.97      0.96      0.97        95
weighted avg       0.97      0.97      0.97        95


Feature Importance:
        feature  importance
1        height    0.295951
2  total_pixels    0.194837
4     color_std    0.158502
0         width    0.145348
3  text_density    0.112306
5  edge_density    0.093056


In [46]:
import shutil

figures_dir = Path("../data/processed/adobe_extracted")
good_img_dir = Path("../data/processed/good_figures")
good_img_dir.mkdir(exist_ok=True)

for img in tqdm(list(Path("../data/processed/tmp_figures").iterdir())):
    if not img.is_file():
        continue

    venue, part = img.name.split("_fileout")
    out_dir = Path(f"../data/processed/good_figures/{venue}")
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"fileout{part}"
    img_properties = photo_properties(img)
    if predict_image_quality(img_properties):
        shutil.copy2(img, out_path)
    else:
        pass

  0%|          | 0/7757 [00:00<?, ?it/s]

100%|██████████| 7757/7757 [1:23:19<00:00,  1.55it/s]  


In [ ]:
for img in Path("../data/processed/train").iterdir():
    venue, _, part = img.name.split("_fileout")
    out_path = Path(f"../data/processed/good_figures/{venue}/fileout{part}")

In [32]:
list((venue / "figures").iterdir())

[]

In [3]:
df["good_img"].value_counts()

good_img
True    473
Name: count, dtype: int64

In [23]:
from pathlib import Path
from tqdm import tqdm

# Create output directory
output_dir = Path("../data/processed/tmp_figures")
output_dir.mkdir(exist_ok=True)

# Iterate through company directories
adobe_dir = Path("../data/processed/adobe_extracted")
for company_dir in tqdm(list(adobe_dir.iterdir())):
    if not company_dir.is_dir() or company_dir.name.startswith(
        "."
    ):  # Skip .DS_Store etc
        continue

    figures_dir = company_dir / "figures"
    if not figures_dir.exists():
        continue

    # Copy each figure with company name prefix
    for fig in figures_dir.iterdir():
        if fig.is_file():
            new_name = f"{company_dir.name}_{fig.name}"
            fig.rename(output_dir / new_name)

100%|██████████| 281/281 [00:01<00:00, 148.84it/s]


In [22]:
for pdf_dir in tqdm(
    list(Path("../data/processed/adobe_extracted").iterdir()), leave=False
):
    filter_photos(pdf_dir / "figures")

  0%|          | 0/281 [00:00<?, ?it/s]

Error processing fileoutpart11.png: Can only use .str accessor with string values!


  3%|▎         | 8/281 [01:52<1:25:28, 18.78s/it]

Error processing fileoutpart21.png: Can only use .str accessor with string values!


Error processing fileoutpart9.png: Can only use .str accessor with string values!


...

Error processing fileoutpart41.png: Can only use .str accessor with string values!
Error processing fileoutpart12.png: Can only use .str accessor with string values!
Error processing fileoutpart8.png: Can only use .str accessor with string values!


Error processing fileoutpart10.png: Can only use .str accessor with string values!


100%|██████████| 26/26 [00:12<00:00,  2.06it/s]


Error processing fileoutpart7.png: Can only use .str accessor with string values!


NotADirectoryError: [Errno 20] Not a directory: '../data/processed/adobe_extracted/.DS_Store/figures_accepted'

In [8]:
image_path = "../data/processed/adobe_extracted/ Monserate Winery (11)-merged/figures/fileoutpart2.png"

In [9]:
import pandas as pd

df = pd.DataFrame(columns=["img", "text_density", "color_std", "edge_density"])

for img in tqdm(
    list(
        Path(
            "../data/processed/adobe_extracted/ Monserate Winery (11)-merged/figures"
        ).iterdir()
    )
):
    properties = photo_properties(img)
    df = pd.concat(
        [
            df,
            pd.DataFrame(
                {
                    "img": img.name,
                    **properties,
                },
                index=[len(df)],
            ),
        ],
    )
df

  0%|          | 0/77 [00:00<?, ?it/s]/tmp/ipykernel_18344/4042842102.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(
 14%|█▍        | 11/77 [00:06<00:20,  3.22it/s]

Error processing fileoutpart4.png: Can only use .str accessor with string values!


 57%|█████▋    | 44/77 [00:27<00:12,  2.69it/s]

Error processing fileoutpart29.png: Can only use .str accessor with string values!


 69%|██████▉   | 53/77 [00:35<00:17,  1.37it/s]

Error processing fileoutpart25.png: Can only use .str accessor with string values!


100%|██████████| 77/77 [00:48<00:00,  1.59it/s]


,img,text_density,color_std,edge_density,width,height,total_pixels
0,fileoutpart24.png,0.000000,64.603376,0.215412,1219.0,1310.0,1596890.0
1,fileoutpart56.png,0.000000,64.181394,0.041583,841.0,1056.0,888096.0
2,fileoutpart78.png,0.000000,58.700875,0.035482,1722.0,2299.0,3958878.0
3,fileoutpart71.png,0.000000,59.360535,0.052521,941.0,1725.0,1623225.0
4,fileoutpart31.png,0.000000,65.388021,0.098004,881.0,1281.0,1128561.0
...,...,...,...,...,...,...,...
72,fileoutpart59.png,0.000000,66.462315,0.032408,1032.0,1538.0,1587216.0
73,fileoutpart6.png,0.000000,57.943581,0.057450,1306.0,978.0,1277268.0
74,fileoutpart49.png,0.504513,45.824228,0.074481,400.0,77.0,30800.0
75,fileoutpart57.png,0.513300,46.333079,0.073534,401.0,78.0,31278.0


In [12]:
import itables

itables.show(df)

In [11]:
bad_images = list(
    x.name
    for x in Path(
        "../data/processed/adobe_extracted/ Monserate Winery (11)-merged/figures copy"
    ).iterdir()
)
df["label"] = df["img"].apply(lambda x: "not photo" if x in bad_images else "photo")
df

,img,text_density,color_std,edge_density,width,height,total_pixels,label
0,fileoutpart24.png,0.000000,64.603376,0.215412,1219.0,1310.0,1596890.0,photo
1,fileoutpart56.png,0.000000,64.181394,0.041583,841.0,1056.0,888096.0,photo
2,fileoutpart78.png,0.000000,58.700875,0.035482,1722.0,2299.0,3958878.0,photo
3,fileoutpart71.png,0.000000,59.360535,0.052521,941.0,1725.0,1623225.0,photo
4,fileoutpart31.png,0.000000,65.388021,0.098004,881.0,1281.0,1128561.0,photo
...,...,...,...,...,...,...,...,...
72,fileoutpart59.png,0.000000,66.462315,0.032408,1032.0,1538.0,1587216.0,photo
73,fileoutpart6.png,0.000000,57.943581,0.057450,1306.0,978.0,1277268.0,photo
74,fileoutpart49.png,0.504513,45.824228,0.074481,400.0,77.0,30800.0,not photo
75,fileoutpart57.png,0.513300,46.333079,0.073534,401.0,78.0,31278.0,not photo


In [13]:
(text_density, color_std, edge_density)

(0.00040239210657175013, np.float64(25.11696419010659), 0.0467761740253169)

In [6]:
import os

os.listdir("../data/processed/adobe_extracted/ Monserate Winery (11)-merged")

['tables', 'figures', 'structuredData.json']